# Importing data and library

In [1]:
# Core Data Manipulation
library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(tibble)
library(forcats)

# Visualization and Tables
library(ggplot2)
library(RColorBrewer)
library(gt)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




# Data import

In [2]:
geral_results <- read.csv("results/STR_vs_scRNA_overlap.csv")
print(geral_results)

    gene_name         cell_type      LogFC        Pvalue   abs_res
1       CDH12     cell_Neuronal   36.48661  1.339790e-58 61.073650
2       CDH12   cell_Epithelial -243.38303  1.641085e-31 61.073650
3       CDH12     cell_Neuronal -152.47242  7.173388e-06 61.073650
4       CDH12        cell_Glial  -65.74969  1.362056e-12 61.073650
5       CDH12     cell_Neuronal -284.09865  1.589222e-03 61.073650
6       CERS6  cell_Macrophages  -13.45123  4.281426e-02 58.468964
7       CERS6    cell_Monocytes   31.61799  1.180799e-02 58.468964
8      R3HDM1   cell_Epithelial  327.89673  3.490684e-08 47.551678
9      NKAIN2        cell_Glial   77.76291  3.166906e-02 43.708304
10     NKAIN2     cell_Neuronal  146.59276 8.648638e-197 43.708304
11     NKAIN2     cell_Neuronal -167.37850  0.000000e+00 43.708304
12     NKAIN2   cell_Epithelial  248.54850  2.549227e-08 43.708304
13     NKAIN2   cell_Fibroblast -426.74502  1.062966e-05 43.708304
14     NKAIN2     cell_Neuronal -295.29923  4.092604e-03 43.70

# Bubble Plot

dbscan signal per tissues

In [3]:
# --- 1. Data Cleaning & Label Formatting ---
# Mapping internal cell type names to publication-quality English labels
cell_type_lookup <- c(
  "cell_T" = "T cells", "cell_B" = "B cells", "cell_SmoothMuscle" = "Smooth Muscle cells",
  "cell_RBC" = "Red Blood cells", "cell_Neutrophils" = "Neutrophils", "cell_Neuronal" = "Neuronal cells",
  "cell_Monocytes" = "Monocytes", "cell_Monoctyes" = "Monocytes", 
  "cell_Mast" = "Mast cells", "cell_Macrophages" = "Macrophages",
  "cell_Fibroblasts" = "Fibroblasts", "cell_Fibroblast" = "Fibroblasts",
  "cell_Epithelial" = "Epithelial cells", "cell_Endothelial" = "Endothelial cells",
  "cell_Glial" = "Glial cells", "cell_Brain" = "Brain cells"
)

df_bubble <- geral_results %>%
  filter(!is.na(STRs_ID), !is.na(LogFC)) %>%
  mutate(
    # Extract motif and size from STRs_ID (e.g., "AT:10")
    str_variant = str_extract(STRs_ID, "[^:]+:[^:]+$"),
    
    # Create the desired label: GENE (Motif:Size)
    gene_variant_label = paste0(gene_name, " (", str_variant, ")"),
    
    # Dynamic cleaning of cell types using the lookup table
    cell_type_clean = case_when(
      cell_type %in% names(cell_type_lookup) ~ cell_type_lookup[cell_type],
      TRUE ~ str_to_title(str_remove(cell_type, "cell_"))
    ),
    
    abs_res = abs(as.numeric(abs_res)),
    LogFC = as.numeric(LogFC)
  ) %>%
  filter(is.finite(LogFC)) %>%
  # Sort alphabetically by gene name to avoid a messy X-axis
  mutate(gene_variant_label = fct_reorder(gene_variant_label, gene_name))

# --- 2. Safety Checks for Color Scales ---
# Ensures the scale limits are valid even with a narrow data range
fill_limits <- range(df_bubble$LogFC, na.rm = TRUE)
if (fill_limits[1] == fill_limits[2]) fill_limits <- c(fill_limits[1]-0.1, fill_limits[2]+0.1)

# --- 3. Bubble Plot Creation ---
p_bubble <- ggplot(df_bubble, aes(x = gene_variant_label, y = cell_type_clean)) +
  # Geometry with transparency and thin strokes to reduce overlap perception
  geom_point(aes(size = abs_res, fill = LogFC), 
             shape = 21, color = "black", stroke = 0.3, alpha = 0.7) +
  
  # Using facet_grid with space="free_x" for proportional gene spacing per tissue
  facet_grid(. ~ source_tissue, scales = "free_x", space = "free_x") + 
  
  scale_fill_gradientn(
    colors = rev(RColorBrewer::brewer.pal(11, "RdBu")),
    limits = fill_limits,
    name = "Log2 FC"
  ) +
  
  # Adjusted range to prevent large bubbles from overcrowding neighbors
  scale_size_continuous(
    range = c(4, 15), 
    name = "STR Intensity\n(abs_res)"
  ) +
  
  # Expand Y-axis to provide more "breathing room" between cell type rows
  scale_y_discrete(expand = expansion(add = 0.4)) + 
  
  theme_minimal() +
  labs(
    title = "Impact of STR Outliers by Gene and Variant",
    subtitle = "Size: Intensity | Color: Log2 FC direction",
    x = "Gene & STR Variant (Motif:Size)", 
    y = "Cell Type"
  ) +
  theme(
    # Fine-tuning X-axis text appearance
    axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, 
                               size = 10, face = "bold.italic"),
    axis.text.y = element_text(face = "bold", size = 12, color = "black"),
    
    # Spacing between tissue blocks
    panel.spacing.x = unit(1.2, "lines"),
    
    # Aesthetic styling for panel headers
    strip.background = element_rect(fill = "grey92", color = "white"),
    strip.text = element_text(face = "bold", size = 10),
    
    # Soft gridlines for horizontal readability
    panel.grid.major.y = element_line(color = "grey95"),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    
    plot.margin = margin(15, 15, 15, 15)
  )

# --- 4. Export with Optimized Dimensions ---
# Increasing the height here prevents crowding of cell types in the final file
ggsave("results/STR_Impact_Final_Optimized.png", p_bubble, 
        width = 18, height = 11, dpi = 400, bg = "white")

Warning message:
“There were 10 warnings in `mutate()`.
The first warning was:
ℹ In argument: `gene_variant_label = fct_reorder(gene_variant_label,
  gene_name)`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 9 remaining warnings.”


table visualization of dbscan per tissues

In [4]:
# --- 1. Data Processing ---
table_summary <- geral_results %>%
  mutate(
    # Ensure all metric columns are numeric
    across(c(Pvalue, LogFC, abs_res, allele1_est, allele2_est, depth), as.numeric),
    # Calculate Median Allele instead of using Allele 1
    median_allele = (allele1_est + allele2_est) / 2,
    # Clean cell type names for display
    cell_type = str_remove(cell_type, "cell_") %>% str_replace_all("_", " "),
    # Standardize group names to Title Case
    group = str_to_title(group)
  ) %>%
  group_by(STRs_ID, gene_name, source_tissue, cell_type, group) %>%
  summarise(
    n_obs = n(), # Count observations per group
    Pvalue = min(Pvalue, na.rm = TRUE),
    LogFC = mean(LogFC[is.finite(LogFC)], na.rm = TRUE),
    abs_res = mean(abs_res, na.rm = TRUE),
    allele2 = mean(allele2_est, na.rm = TRUE),
    depth = mean(depth, na.rm = TRUE),
    median_allele = mean(median_allele, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  # Pivot to Wide format for side-by-side Case vs. Control comparison
  pivot_wider(
    names_from = group,
    values_from = c(n_obs, abs_res, allele2, depth, median_allele),
    names_glue = "{.value}_{group}"
  ) %>%
  arrange(source_tissue, Pvalue)

# --- 2. GT Table Creation ---
gt_final_table <- table_summary %>%
  gt(groupname_col = "source_tissue") %>%
  tab_header(
    title = md("**STR Outlier Impact: Full Quantitative Analysis**"),
  ) %>%
  cols_label(
    STRs_ID = md("**STR ID**"), 
    gene_name = md("**Gene**"),
    cell_type = md("**Cell Type**"), 
    Pvalue = md("*p*-val"), 
    LogFC = md("**Log2 FC**"),
    # Case group headers
    n_obs_Case = md("**n**"), 
    abs_res_Case = "Abs. Res", 
    allele2_Case = "Allele 2", 
    depth_Case = "Depth", 
    median_allele_Case = "Median Allele",
    # Control group headers
    n_obs_Control = md("**n**"), 
    abs_res_Control = "Abs. Res", 
    allele2_Control = "Allele 2", 
    depth_Control = "Depth", 
    median_allele_Control = "Median Allele"
  ) %>%
  
  # Spanners for visual organization
  tab_spanner(label = md("**Case Group**"), columns = ends_with("_Case")) %>%
  tab_spanner(label = md("**Control Group**"), columns = ends_with("_Control")) %>%
  
  # --- THE MASTER TOUCH: CENTRAL ALIGNMENT ---
  cols_align(
    align = "center",
    columns = everything() # Center all columns in the table
  ) %>%
  
  # Numerical Formatting
  fmt_scientific(columns = Pvalue, decimals = 2) %>%
  fmt_number(columns = contains("abs_res") | contains("LogFC") | contains("median_allele"), decimals = 2) %>%
  fmt_number(columns = contains("allele2") | contains("depth"), decimals = 1) %>%
  fmt_integer(columns = contains("n_obs")) %>%
  fmt_missing(columns = everything(), missing_text = "-") %>%
  
  # General Aesthetics
  tab_options(
    row_group.background.color = "grey96",
    row_group.font.weight = "bold",
    column_labels.font.weight = "bold",
    table.font.size = px(10),
    table.width = pct(100)
  )

# --- 3. Save Output ---
if(!dir.exists("results")) dir.create("results") # Ensure directory exists
gtsave(gt_final_table, "results/STR_Centered_Quantitative_Table.html")

Warning message:
“Since gt v0.6.0 `fmt_missing()` is deprecated and will soon be removed.
ℹ Use `sub_missing()` instead.
This warning is displayed once every 8 hours.”


 Integration of data filtred in STRs_filter step and scRNA-seq analysis

Function

In [5]:
# ==============================================================================
# MASTER FUNCTION: INTEGRATED STR & scRNA-seq ANALYSIS
# ==============================================================================
library(tidyverse)
library(gt)

analyze_str_expression <- function(str_path, 
                                   scrna_path = "../../6_scovid_data/smerged_covid.csv",
                                   output_prefix = "STR_Analysis",
                                   plot_title = "Impact of STR Outliers by Gene and Variant",
                                   color_palette = "RdBu") {
  
  cat(sprintf("\n--- STARTING ANALYSIS: %s ---\n", plot_title))
  
  # 1. Load Data
  if (!file.exists(str_path)) stop(paste("STR file not found at:", str_path))
  if (!file.exists(scrna_path)) stop(paste("scRNA-seq file not found at:", scrna_path))
  
  df_str <- read_csv2(str_path, show_col_types = FALSE)
  raw_scrna <- read_csv(scrna_path, show_col_types = FALSE)
  
  # === [DEBUG STEP 1: INPUT VALIDATION] ===
  cat("\n🔍 [DEBUG 1] Files loaded successfully!\n")
  cat("  -> Rows in STR file:", nrow(df_str), "\n")
  cat("  -> Rows in scRNA-seq file:", nrow(raw_scrna), "\n")
  cat("  -> Does 'GEO_ID' originally exist in STR file?:", "GEO_ID" %in% colnames(df_str), "\n")
  cat("  -> Does 'GEO_ID' originally exist in scRNA-seq file?:", "GEO_ID" %in% colnames(raw_scrna), "\n")
  cat("  -> Columns detected in STR:", paste(colnames(df_str), collapse = ", "), "\n")
  cat("  -> First 10 columns in scRNA-seq:", paste(head(colnames(raw_scrna), 10), collapse = ", "), "...\n\n")
  # ===============================================

  # 2. Complete Cell Type Lookup
  cell_type_lookup <- c(
    "cell_T" = "T cells", "cell_B" = "B cells", "cell_SmoothMuscle" = "Smooth Muscle cells",
    "cell_RBC" = "Red Blood cells", "cell_Neutrophils" = "Neutrophils", "cell_Neuronal" = "Neuronal cells",
    "cell_Monocytes" = "Monocytes", "cell_Monoctyes" = "Monocytes", 
    "cell_Mast" = "Mast cells", "cell_Macrophages" = "Macrophages",
    "cell_Fibroblasts" = "Fibroblasts", "cell_Fibroblast" = "Fibroblasts",
    "cell_Epithelial" = "Epithelial cells", "cell_Endothelial" = "Endothelial cells",
    "cell_Glial" = "Glial cells", "cell_Brain" = "Brain cells"
  )

  # 3. Data Integration & Cleaning (TYPE-SAFE SHIELD)
  
  # Standardize gene names to avoid mismatches
  raw_scrna_clean <- raw_scrna %>%
    mutate(`Gene Symbol` = str_trim(str_to_upper(as.character(`Gene Symbol`))))
    
  df_str_clean <- df_str %>%
    mutate(gene_name = str_trim(str_to_upper(as.character(gene_name))))

  # Perform join/integration
  df_integrated <- raw_scrna_clean %>%
    inner_join(df_str_clean, by = c("Gene Symbol" = "gene_name"), relationship = "many-to-many") %>%
    rename(gene_name = `Gene Symbol`)

  # === [RUNTIME DEBUG] ===
  cat(sprintf("🔍 Merged rows after Join: %d\n", nrow(df_integrated)))
  # =======================

  if (nrow(df_integrated) == 0) {
    cat("X No gene matches found between both files. Aborting.\n")
    return(NULL)
  }

  # Safe column cleaning and transformation
  df_integrated <- df_integrated %>%
    mutate(
      # Ensure source_tissue and group are not NA or empty strings
      source_tissue = if_else(is.na(source_tissue) | source_tissue == "", "Unknown Tissue", as.character(source_tissue)),
      group = if_else(is.na(group) | group == "", "Unknown Group", as.character(group)),
      
      # Safe STR variant extraction
      str_variant = if_else(!is.na(STRs_ID), str_extract(STRs_ID, "[^:]+:[^:]+$"), "Unknown_Variant"),
      gene_variant_label = paste0(gene_name, " (", str_variant, ")"),
      
      # Cell type handling
      cell_type_clean = case_when(
        cell_type %in% names(cell_type_lookup) ~ cell_type_lookup[cell_type],
        is.na(cell_type) | cell_type == "" ~ "Unknown Cell Type",
        TRUE ~ str_to_title(str_remove(as.character(cell_type), "cell_"))
      ),
      
      # Numeric conversion forcing comma-to-dot replacement if necessary
      LogFC = as.numeric(str_replace_all(as.character(LogFC), ",", ".")),
      Pvalue = as.numeric(str_replace_all(as.character(Pvalue), ",", ".")),
      
      # Intensity handling
      intensity = if ("abs_res" %in% colnames(.)) {
        abs(as.numeric(str_replace_all(as.character(.data[["abs_res"]]), ",", ".")))
      } else if ("abs_residual" %in% colnames(.)) {
        abs(as.numeric(str_replace_all(as.character(.data[["abs_residual"]]), ",", ".")))
      } else {
        1
      }
    ) %>%
    # Replace invalid LogFC with 0 to prevent ggplot from crashing, keeping rows alive for the CSV
    mutate(LogFC = if_else(is.finite(LogFC), LogFC, 0)) %>%
    mutate(gene_variant_label = fct_reorder(gene_variant_label, gene_name))

  # 4. Color Scale Safety Limits
  fill_limits <- range(df_integrated$LogFC, na.rm = TRUE)
  if (fill_limits[1] == fill_limits[2]) fill_limits <- c(fill_limits[1]-0.1, fill_limits[2]+0.1)

  # 5. Generate Bubble Plot
  cat("[PROCESS] Generating Bubble Plot...\n")
  p <- ggplot(df_integrated, aes(x = gene_variant_label, y = cell_type_clean)) +
    geom_point(aes(size = intensity, fill = LogFC), 
               shape = 21, color = "black", stroke = 0.3, alpha = 0.7) +
    facet_grid(. ~ source_tissue, scales = "free_x", space = "free_x") + 
    scale_fill_gradientn(colors = rev(RColorBrewer::brewer.pal(11, color_palette)), limits = fill_limits, name = "Log2 FC") +
    scale_size_continuous(range = c(4, 15), name = "STR Intensity\n(abs_res)") +
    scale_y_discrete(expand = expansion(add = 0.4)) + 
    theme_minimal() +
    labs(title = plot_title, subtitle = "Size: Intensity | Color: Log2 FC direction", x = "Gene & STR Variant (Motif:Size)", y = "Cell Type") +
    theme(
      axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 10, face = "bold.italic"),
      axis.text.y = element_text(face = "bold", size = 12, color = "black"),
      panel.spacing.x = unit(1.2, "lines"),
      strip.background = element_rect(fill = "grey92", color = "white"),
      strip.text = element_text(face = "bold", size = 10),
      panel.grid.major.y = element_line(color = "grey95"),
      panel.grid.major.x = element_blank(),
      plot.margin = margin(15, 15, 15, 15)
    )

  # 6. Generate Quantitative Summary Table
  cat("[PROCESS] Generating Quantitative Table...\n")
  table_data <- df_integrated %>%
    mutate(
      across(any_of(c("Pvalue", "LogFC", "intensity", "allele1_est", "allele2_est", "depth")), as.numeric),
      median_allele = if("allele1_est" %in% colnames(.) & "allele2_est" %in% colnames(.)) (allele1_est + allele2_est) / 2 else NA,
      group = if("group" %in% colnames(.)) str_to_title(group) else "Unknown"
    ) %>%
    group_by(STRs_ID, gene_name, source_tissue, cell_type_clean, group) %>%
    summarise(
      n_obs = n(),
      Pvalue = min(Pvalue, na.rm = TRUE),
      LogFC = mean(LogFC, na.rm = TRUE),
      abs_res = mean(intensity, na.rm = TRUE),
      allele2 = if("allele2_est" %in% colnames(.)) mean(allele2_est, na.rm = TRUE) else NA,
      depth = if("depth" %in% colnames(.)) mean(depth, na.rm = TRUE) else NA,
      median_allele = mean(median_allele, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    pivot_wider(names_from = group, values_from = c(n_obs, abs_res, allele2, depth, median_allele), names_glue = "{.value}_{group}")

  gt_table <- table_data %>%
    gt(groupname_col = "source_tissue") %>%
    tab_header(title = md(paste0("**Quantitative Analysis:** ", plot_title))) %>%
    cols_label(STRs_ID = md("**STR ID**"), gene_name = md("**Gene**"), cell_type_clean = md("**Cell Type**"), Pvalue = md("*p*-val"), LogFC = md("**Log2 FC**")) %>%
    tab_spanner(label = md("**Case Group**"), columns = contains("_Case")) %>%
    tab_spanner(label = md("**Control Group**"), columns = contains("_Control")) %>%
    cols_align(align = "center", columns = everything()) %>%
    fmt_scientific(columns = Pvalue, decimals = 2) %>%
    fmt_number(columns = contains("abs_res") | contains("LogFC") | contains("median_allele"), decimals = 2) %>%
    fmt_missing(columns = everything(), missing_text = "-") %>%
    tab_options(row_group.background.color = "grey96", row_group.font.weight = "bold", table.font.size = px(10))

  # 7. Export
  if(!dir.exists("results")) dir.create("results")
  ggsave(sprintf("results/%s_Plot_Optimized.png", output_prefix), p, width = 18, height = 11, dpi = 400, bg = "white")
  gtsave(gt_table, sprintf("results/%s_Table.html", output_prefix))
  
  # === [DEBUG STEP 7: CSV EXPORTATION] ===
  cat("\n🔍 [DEBUG 7] Attempting to export CSV with GEO_ID...\n")
  if ("GEO_ID" %in% colnames(df_integrated)) {
    cat("  -> [SUCCESS] 'GEO_ID' column found! Saving file...\n")
    df_integrated %>%
      relocate(GEO_ID) %>%
      write_csv(sprintf("results/%s_GEO_Data.csv", output_prefix))
    cat(sprintf("  -> [OK] File written to: results/%s_GEO_Data.csv\n", output_prefix))
  } else {
    cat("  -> [WARNING] Aborted: 'GEO_ID' does NOT exist in the final dataset. Please check column headers.\n")
  }
  # =======================================
  
  cat(sprintf("\n✓ Process completed for prefix: %s\n", output_prefix))
  return(list(plot = p, table = gt_table))
}

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ lubridate 1.9.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Top 10 Outliers

In [6]:
res1 <- analyze_str_expression(
  str_path = "../7.3_STRs_filter/results/top_10_unique_loci.csv",
  output_prefix = "Top10_Outliers",
  plot_title = "Top 10 STR Outliers by Residual Magnitude"
)


--- STARTING ANALYSIS: Top 10 STR Outliers by Residual Magnitude ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.




🔍 [DEBUG 1] Files loaded successfully!
  -> Rows in STR file: 434 
  -> Rows in scRNA-seq file: 8349 
  -> Does 'GEO_ID' originally exist in STR file?: FALSE 
  -> Does 'GEO_ID' originally exist in scRNA-seq file?: TRUE 
  -> Columns detected in STR: STRs_ID, gene_name, group, allele1_est, allele2_est, outlier_residuals, abs_res, depth, chrom, start, region 
  -> First 10 columns in scRNA-seq: Gene Symbol, Ensembl ID, Pvalue, LogFC, cell_type, source_tissue, GEO_ID, origin_filename, GEO, Dataset ...

🔍 Merged rows after Join: 0
X No gene matches found between both files. Aborting.


Major alelle without overlap

In [7]:
res2 <- analyze_str_expression(
  str_path = "../7.3_STRs_filter/results/covid_targeted_no_overlap_allele2.csv",
  output_prefix = "overlap_allele2",
  plot_title = "STRs without overlap in the major allele between groups"
)


--- STARTING ANALYSIS: STRs without overlap in the major allele between groups ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.




🔍 [DEBUG 1] Files loaded successfully!
  -> Rows in STR file: 49 
  -> Rows in scRNA-seq file: 8349 
  -> Does 'GEO_ID' originally exist in STR file?: FALSE 
  -> Does 'GEO_ID' originally exist in scRNA-seq file?: TRUE 
  -> Columns detected in STR: STRs_ID, gene_name, group, target_metric_allele2, allele1_est, allele2_est, outlier_residuals, depth, chrom, start, region 
  -> First 10 columns in scRNA-seq: Gene Symbol, Ensembl ID, Pvalue, LogFC, cell_type, source_tissue, GEO_ID, origin_filename, GEO, Dataset ...

🔍 Merged rows after Join: 306


Warning message:
“There were 7 warnings in `mutate()`.
The first warning was:
ℹ In argument: `gene_variant_label = fct_reorder(gene_variant_label,
  gene_name)`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 6 remaining warnings.”


[PROCESS] Generating Bubble Plot...
[PROCESS] Generating Quantitative Table...

🔍 [DEBUG 7] Attempting to export CSV with GEO_ID...
  -> [SUCCESS] 'GEO_ID' column found! Saving file...
  -> [OK] File written to: results/overlap_allele2_GEO_Data.csv

✓ Process completed for prefix: overlap_allele2


Mean alelle without overlap

In [8]:
res_a2 <- analyze_str_expression(
  str_path        = "../7.3_STRs_filter/results/covid_targeted_no_overlap_mean_allele.csv",
  output_prefix   = "overlap_mean_allele",
  plot_title      = "STRs without overlap in the mean allele between groups",
  color_palette   = "RdBu" 
)


--- STARTING ANALYSIS: STRs without overlap in the mean allele between groups ---


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.




🔍 [DEBUG 1] Files loaded successfully!
  -> Rows in STR file: 50 
  -> Rows in scRNA-seq file: 8349 
  -> Does 'GEO_ID' originally exist in STR file?: FALSE 
  -> Does 'GEO_ID' originally exist in scRNA-seq file?: TRUE 
  -> Columns detected in STR: STRs_ID, gene_name, group, mean_allele_calc, allele1_est, allele2_est, outlier_residuals, depth, chrom, start, region 
  -> First 10 columns in scRNA-seq: Gene Symbol, Ensembl ID, Pvalue, LogFC, cell_type, source_tissue, GEO_ID, origin_filename, GEO, Dataset ...

🔍 Merged rows after Join: 346


Warning message:
“There were 7 warnings in `mutate()`.
The first warning was:
ℹ In argument: `gene_variant_label = fct_reorder(gene_variant_label,
  gene_name)`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 6 remaining warnings.”


[PROCESS] Generating Bubble Plot...
[PROCESS] Generating Quantitative Table...

🔍 [DEBUG 7] Attempting to export CSV with GEO_ID...
  -> [SUCCESS] 'GEO_ID' column found! Saving file...
  -> [OK] File written to: results/overlap_mean_allele_GEO_Data.csv

✓ Process completed for prefix: overlap_mean_allele
